# Pertemuan 7 — Constraint Satisfaction Problem (CSP) dan Pengantar Perencanaan
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Pada pencarian biasa, status adalah "kotak hitam": algoritma hanya bisa bertanya *apa aksinya* dan *apakah ini tujuan*. Pada **CSP**, status memiliki struktur: sekumpulan **variabel** yang harus diberi **nilai** tanpa melanggar **batasan**. Struktur itu memungkinkan algoritma umum yang jauh lebih cerdas.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. memformulasikan masalah sebagai CSP (variabel, domain, batasan);
2. mengimplementasikan backtracking search dan memperkuatnya dengan MRV, degree heuristic, LCV, forward checking, dan AC-3;
3. menjelaskan gagasan dasar perencanaan (*planning*) sebagai pencarian atas status berbentuk fakta.

Notebook ini mendukung asesmen **A3 — Tugas Metode AI (bobot 10%)**.

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P7 dikerjakan oleh: {NAMA} ({NIM})")

import itertools, random, time
from collections import deque
import pandas as pd, networkx as nx, matplotlib.pyplot as plt

## 1. Formulasi CSP: mewarnai peta Sumatera

Warnai sepuluh provinsi di Pulau Sumatera dan sekitarnya dengan **tiga warna** sehingga provinsi yang berbatasan darat tidak sewarna.

- **Variabel**: sepuluh provinsi
- **Domain**: {merah, hijau, biru} untuk setiap variabel
- **Batasan**: untuk setiap pasangan provinsi bertetangga, warnanya harus berbeda

Batasan yang melibatkan dua variabel disebut **batasan biner**, dan dapat digambar sebagai **graf batasan**.

In [ ]:
TETANGGA_SUMATERA = {
    "Aceh": ["Sumut"],
    "Sumut": ["Aceh", "Riau", "Sumbar"],
    "Riau": ["Sumut", "Sumbar", "Jambi"],
    "Sumbar": ["Sumut", "Riau", "Jambi", "Bengkulu"],
    "Jambi": ["Riau", "Sumbar", "Bengkulu", "Sumsel"],
    "Bengkulu": ["Sumbar", "Jambi", "Sumsel", "Lampung"],
    "Sumsel": ["Jambi", "Bengkulu", "Lampung"],
    "Lampung": ["Bengkulu", "Sumsel"],
    "Kepri": [],                 # provinsi kepulauan: tidak punya batas darat
    "Babel": [],
}
WARNA = ["merah", "hijau", "biru"]

class CSP:
    """CSP dengan batasan biner. batasan(X, x, Y, y) bernilai True bila X = x dan Y = y boleh bersamaan."""
    def __init__(self, variabel, domain, tetangga, batasan):
        self.variabel = list(variabel)
        self.domain = {v: list(domain[v]) for v in variabel}
        self.tetangga = tetangga
        self.batasan = batasan
    def konsisten(self, var, nilai, penugasan):
        return all(self.batasan(var, nilai, t, penugasan[t]) for t in self.tetangga[var] if t in penugasan)

def beda(X, x, Y, y):
    return x != y

peta = CSP(TETANGGA_SUMATERA, {p: WARNA for p in TETANGGA_SUMATERA}, TETANGGA_SUMATERA, beda)
print("Jumlah variabel:", len(peta.variabel))
print("Jumlah batasan :", sum(len(t) for t in TETANGGA_SUMATERA.values()) // 2)
print("Jumlah penugasan lengkap yang mungkin: 3^10 =", 3 ** 10)

In [ ]:
POSISI_PROV = {"Aceh": (0, 6), "Sumut": (1.2, 4.7), "Riau": (3.0, 3.9), "Sumbar": (1.7, 3.0), "Jambi": (3.5, 2.3),
               "Bengkulu": (2.4, 1.0), "Sumsel": (4.4, 1.1), "Lampung": (4.6, -0.4), "Kepri": (5.2, 4.4), "Babel": (5.9, 1.6)}
KODE = {"merah": "#D1495B", "hijau": "#6A994E", "biru": "#3D5A80", None: "#E6F2F1"}

def gambar_csp(tetangga, posisi, penugasan=None, judul="Graf batasan"):
    Gc = nx.Graph(); Gc.add_nodes_from(tetangga)
    Gc.add_edges_from((a, b) for a in tetangga for b in tetangga[a])
    penugasan = penugasan or {}
    plt.figure(figsize=(6.5, 6))
    nx.draw_networkx_nodes(Gc, posisi, node_size=2300, edgecolors="#12263A",
                           node_color=[KODE.get(penugasan.get(v), "#F2A024") for v in Gc])
    nx.draw_networkx_edges(Gc, posisi, edge_color="#8A9AA5", width=1.6)
    nx.draw_networkx_labels(Gc, posisi, font_size=9, font_color="white" if penugasan else "#12263A")
    plt.title(judul); plt.axis("off"); plt.show()

gambar_csp(TETANGGA_SUMATERA, POSISI_PROV)

## 2. Backtracking search
Gagasan dasarnya: beri nilai pada **satu variabel per langkah**. Bila nilai itu melanggar batasan, coba nilai lain. Bila semua nilai gagal, **mundur** (*backtrack*) ke variabel sebelumnya.

Fungsi di bawah dibuat dapat dikonfigurasi supaya kita bisa menyalakan dan mematikan setiap perbaikan lalu mengukur pengaruhnya:

| Parameter | Pilihan | Pertanyaan yang dijawab |
|---|---|---|
| `pilih_variabel` | `urut`, `mrv` | variabel mana yang diisi berikutnya? |
| `urut_nilai` | `urut`, `lcv` | nilai mana yang dicoba lebih dulu? |
| `inferensi` | `tanpa`, `forward_checking` | bisakah kegagalan dideteksi lebih awal? |

In [ ]:
def urut(csp, penugasan, domain):
    """Pilih variabel pertama yang belum diberi nilai (tanpa heuristik)."""
    return next(v for v in csp.variabel if v not in penugasan)

def mrv(csp, penugasan, domain):
    """Minimum Remaining Values: pilih variabel dengan sisa nilai sah paling sedikit.
    Bila seri, pilih yang terlibat batasan paling banyak dengan variabel belum terisi (degree heuristic)."""
    belum = [v for v in csp.variabel if v not in penugasan]
    def sisa(v): return sum(1 for x in domain[v] if csp.konsisten(v, x, penugasan))
    def derajat(v): return sum(1 for t in csp.tetangga[v] if t not in penugasan)
    return min(belum, key=lambda v: (sisa(v), -derajat(v)))

def nilai_urut(csp, var, penugasan, domain):
    return list(domain[var])

def lcv(csp, var, penugasan, domain):
    """Least Constraining Value: dahulukan nilai yang paling sedikit menghapus pilihan tetangga."""
    def konflik(x):
        return sum(1 for t in csp.tetangga[var] if t not in penugasan
                   for y in domain[t] if not csp.batasan(var, x, t, y))
    return sorted(domain[var], key=konflik)

def tanpa(csp, var, nilai, penugasan, domain):
    return domain

def forward_checking(csp, var, nilai, penugasan, domain):
    """Hapus dari domain tetangga semua nilai yang bentrok dengan var = nilai.
    Mengembalikan domain baru, atau None bila ada domain yang menjadi kosong."""
    baru = {v: list(d) for v, d in domain.items()}
    baru[var] = [nilai]
    for t in csp.tetangga[var]:
        if t not in penugasan:
            baru[t] = [y for y in baru[t] if csp.batasan(var, nilai, t, y)]
            if not baru[t]:
                return None
    return baru

def backtracking(csp, pilih_variabel=urut, urut_nilai=nilai_urut, inferensi=tanpa, jejak=False):
    statistik = {"penugasan dicoba": 0, "mundur": 0}
    def rekursi(penugasan, domain):
        if len(penugasan) == len(csp.variabel):
            return penugasan
        var = pilih_variabel(csp, penugasan, domain)
        for nilai in urut_nilai(csp, var, penugasan, domain):
            statistik["penugasan dicoba"] += 1
            if csp.konsisten(var, nilai, penugasan):
                domain_baru = inferensi(csp, var, nilai, {**penugasan, var: nilai}, domain)
                if domain_baru is not None:
                    if jejak: print("  " * len(penugasan) + f"{var} = {nilai}")
                    temuan = rekursi({**penugasan, var: nilai}, domain_baru)
                    if temuan is not None:
                        return temuan
            if jejak: print("  " * len(penugasan) + f"{var} = {nilai} ✗")
        statistik["mundur"] += 1
        return None
    return rekursi({}, csp.domain), statistik

solusi, st = backtracking(peta, jejak=True)
print("\nStatistik:", st)

In [ ]:
gambar_csp(TETANGGA_SUMATERA, POSISI_PROV, solusi, "Salah satu solusi pewarnaan tiga warna")

Peta Sumatera terlalu mudah: urutan variabel yang kebetulan baik membuat backtracking polos pun langsung berhasil. Untuk melihat manfaat heuristik, kita butuh masalah yang lebih sulit.

## 3. Mengukur manfaat heuristik: N-Queens
Letakkan N ratu pada papan N×N sehingga tidak ada yang saling menyerang.
- **Variabel**: `Q1 … QN` (ratu pada kolom ke-i)
- **Domain**: nomor baris 1 … N
- **Batasan**: dua ratu tidak boleh sebaris dan tidak boleh sediagonal

In [ ]:
def buat_nqueens(n):
    variabel = list(range(1, n + 1))
    def tidak_menyerang(X, x, Y, y):
        return x != y and abs(x - y) != abs(X - Y)
    return CSP(variabel, {v: list(range(1, n + 1)) for v in variabel},
               {v: [w for w in variabel if w != v] for v in variabel}, tidak_menyerang)

KONFIGURASI = {
    "backtracking polos": dict(),
    "+ forward checking": dict(inferensi=forward_checking),
    "+ MRV": dict(pilih_variabel=mrv),
    "+ MRV + forward checking": dict(pilih_variabel=mrv, inferensi=forward_checking),
    "+ MRV + FC + LCV": dict(pilih_variabel=mrv, inferensi=forward_checking, urut_nilai=lcv),
}
baris = []
for n in (8, 12, 16):
    for nama, opsi in KONFIGURASI.items():
        t = time.time(); solusi_q, st = backtracking(buat_nqueens(n), **opsi)
        baris.append({"N": n, "konfigurasi": nama, "penugasan dicoba": st["penugasan dicoba"],
                      "mundur": st["mundur"], "waktu (detik)": round(time.time() - t, 3)})
hasil_q = pd.DataFrame(baris)
print(hasil_q.pivot(index="konfigurasi", columns="N", values="penugasan dicoba").loc[list(KONFIGURASI)].to_string())

In [ ]:
pivot = hasil_q.pivot(index="N", columns="konfigurasi", values="penugasan dicoba")[list(KONFIGURASI)]
ax = pivot.plot(marker="o", logy=True, figsize=(8.5, 4.4), color=["#8A9AA5", "#3D5A80", "#F2A024", "#0E7C86", "#D1495B"])
ax.set_ylabel("penugasan dicoba (skala log)"); ax.set_title("N-Queens: pengaruh heuristik pada backtracking")
ax.set_xticks([8, 12, 16]); ax.grid(alpha=0.3); plt.show()

Untuk N = 16, backtracking polos mencoba lebih dari seratus ribu penugasan. Dengan MRV dan forward checking jumlahnya turun menjadi puluhan saja: selisih beberapa tingkat besaran. Catatan: tidak ada heuristik yang selalu menang pada semua masalah, sehingga pilihan konfigurasi harus diuji pada kasusnya.

## 4. Propagasi batasan: AC-3
Forward checking hanya memeriksa tetangga dari variabel yang baru diisi. **Arc consistency** melangkah lebih jauh: variabel `X` disebut *arc-consistent* terhadap `Y` bila setiap nilai di domain `X` punya **pasangan yang sah** di domain `Y`. Algoritma AC-3 memangkas domain sampai seluruh busur konsisten.

Contoh: tiga mata kuliah dengan batasan urutan jam.

In [ ]:
def ac3(csp, domain=None, jejak=False):
    domain = {v: list(d) for v, d in (domain or csp.domain).items()}
    antrean = deque((x, y) for x in csp.variabel for y in csp.tetangga[x])
    while antrean:
        x, y = antrean.popleft()
        dibuang = [vx for vx in domain[x] if not any(csp.batasan(x, vx, y, vy) for vy in domain[y])]
        if dibuang:
            domain[x] = [v for v in domain[x] if v not in dibuang]
            if jejak: print(f"busur ({x}, {y}): buang {dibuang} dari domain {x} → {domain[x]}")
            if not domain[x]:
                return None
            antrean.extend((z, x) for z in csp.tetangga[x] if z != y)
    return domain

# Tiga kuliah harus berurutan: Algoritma < Struktur Data < Kecerdasan Buatan (jam mulai 1..4)
URUTAN = {("Algoritma", "Struktur Data"), ("Struktur Data", "Kecerdasan Buatan")}
def lebih_dulu(X, x, Y, y):
    if (X, Y) in URUTAN: return x < y
    if (Y, X) in URUTAN: return y < x
    return True
kuliah = CSP(["Algoritma", "Struktur Data", "Kecerdasan Buatan"],
             {v: [1, 2, 3, 4] for v in ["Algoritma", "Struktur Data", "Kecerdasan Buatan"]},
             {"Algoritma": ["Struktur Data"], "Struktur Data": ["Algoritma", "Kecerdasan Buatan"],
              "Kecerdasan Buatan": ["Struktur Data"]}, lebih_dulu)
print("Domain awal :", kuliah.domain)
print("Setelah AC-3:", ac3(kuliah, jejak=True))

Tanpa mencoba satu penugasan pun, AC-3 sudah memangkas domain dari 4 nilai menjadi 2 nilai per variabel.

## 5. Kasus terapan: penjadwalan ujian
Enam mata kuliah harus dijadwalkan ke dalam **tiga slot** ujian. Dua mata kuliah tidak boleh di slot yang sama bila ada mahasiswa yang mengambil keduanya. Ada pula batasan tambahan dari dosen.

Ini masalah pewarnaan graf dengan baju lain: mata kuliah = simpul, bentrok = sisi, slot = warna.

In [ ]:
BENTROK = {
    "Kecerdasan Buatan": ["Basis Data", "Jaringan Komputer", "Statistika"],
    "Basis Data": ["Kecerdasan Buatan", "Pemrograman Web", "Statistika"],
    "Jaringan Komputer": ["Kecerdasan Buatan", "Sistem Operasi"],
    "Pemrograman Web": ["Basis Data", "Sistem Operasi"],
    "Sistem Operasi": ["Jaringan Komputer", "Pemrograman Web", "Statistika"],
    "Statistika": ["Kecerdasan Buatan", "Basis Data", "Sistem Operasi"],
}
SLOT = ["Senin pagi", "Senin siang", "Selasa pagi"]
domain_ujian = {mk: list(SLOT) for mk in BENTROK}
# Batasan unary (satu variabel) ditangani dengan memangkas domain sejak awal:
domain_ujian["Statistika"].remove("Senin pagi")            # dosen Statistika berhalangan Senin pagi
domain_ujian["Kecerdasan Buatan"] = ["Senin pagi"]          # ujian AI wajib Senin pagi (ruang lab hanya tersedia saat itu)

ujian = CSP(BENTROK, domain_ujian, BENTROK, beda)
jadwal, st = backtracking(ujian, pilih_variabel=mrv, inferensi=forward_checking)
print("Statistik:", st)
print(pd.Series(jadwal).sort_values().to_string())

In [ ]:
# Verifikasi mandiri: solusi yang baik selalu diperiksa ulang terhadap seluruh batasan
def periksa(csp, penugasan):
    pelanggaran = [(a, b) for a in csp.variabel for b in csp.tetangga[a]
                   if a < b and not csp.batasan(a, penugasan[a], b, penugasan[b])]
    di_luar_domain = [v for v in csp.variabel if penugasan[v] not in csp.domain[v]]
    return pelanggaran, di_luar_domain

print("Pelanggaran batasan biner, nilai di luar domain:", periksa(ujian, jadwal))

def semua_solusi(csp):
    return [dict(zip(csp.variabel, kombinasi)) for kombinasi in itertools.product(*(csp.domain[v] for v in csp.variabel))
            if not periksa(csp, dict(zip(csp.variabel, kombinasi)))[0]]
print("Jumlah seluruh jadwal yang sah:", len(semua_solusi(ujian)))

## 6. Pengantar perencanaan (planning)
Pada perencanaan klasik, status ditulis sebagai **himpunan fakta**, dan setiap aksi memiliki:
- **prasyarat**: fakta yang harus benar sebelum aksi dijalankan;
- **efek tambah** dan **efek hapus**: fakta yang menjadi benar dan menjadi salah sesudahnya.

Inilah gaya representasi **STRIPS/PDDL**. Dengan representasi ini, satu perencana umum dapat dipakai untuk domain apa pun.

Contoh: **dunia balok**. Tiga balok A, B, C. Awalnya C di atas A, sedangkan A dan B di meja. Tujuan: A di atas B, dan B di atas C.

In [ ]:
BALOK = ["A", "B", "C"]

def aksi_sah(status):
    """Menghasilkan (nama aksi, prasyarat, tambah, hapus) untuk semua aksi yang prasyaratnya terpenuhi."""
    daftar = []
    for x in BALOK:
        for dari in BALOK + ["Meja"]:
            for ke in BALOK + ["Meja"]:
                if x in (dari, ke) or dari == ke: continue
                prasyarat = {("di_atas", x, dari), ("bebas", x)} | ({("bebas", ke)} if ke != "Meja" else set())
                tambah = {("di_atas", x, ke)} | ({("bebas", dari)} if dari != "Meja" else set())
                hapus = {("di_atas", x, dari)} | ({("bebas", ke)} if ke != "Meja" else set())
                if prasyarat <= status:
                    daftar.append((f"pindah({x}, {dari} → {ke})", prasyarat, tambah, hapus))
    return daftar

def rencanakan(awal, tujuan):
    """Forward state-space planning dengan BFS: menghasilkan rencana terpendek."""
    awal = frozenset(awal); antrean = deque([(awal, [])]); dikunjungi = {awal}
    while antrean:
        status, rencana = antrean.popleft()
        if tujuan <= status:
            return rencana, len(dikunjungi)
        for nama, _, tambah, hapus in aksi_sah(status):
            baru = frozenset((status - hapus) | tambah)
            if baru not in dikunjungi:
                dikunjungi.add(baru); antrean.append((baru, rencana + [nama]))
    return None, len(dikunjungi)

awal = {("di_atas", "C", "A"), ("di_atas", "A", "Meja"), ("di_atas", "B", "Meja"), ("bebas", "C"), ("bebas", "B")}
tujuan = {("di_atas", "A", "B"), ("di_atas", "B", "C")}
rencana, n = rencanakan(awal, tujuan)
print(f"Rencana ({len(rencana)} langkah, {n} status dijelajahi):")
for i, langkah in enumerate(rencana, 1): print(f"  {i}. {langkah}")

Perhatikan bahwa perencana ini **tidak tahu apa-apa tentang balok**. Ia hanya memanipulasi fakta, prasyarat, dan efek. Mengganti domain (misalnya logistik atau robot gudang) cukup dengan mengganti definisi aksinya.

---
## Tugas A3 — Tugas Metode AI (bobot 10%)

Rincian ketentuan dan rubrik ada pada lembar tugas pertemuan 7. Ringkasannya:

### Bagian A — Pemilihan metode (25%)
Pilih **satu** kasus penjadwalan atau penempatan yang nyata di sekitar Anda (contoh: jadwal piket, jadwal praktikum laboratorium, pembagian kelompok dengan syarat, jadwal pertandingan liga kelas). Tuliskan variabel, domain, dan batasannya, serta jelaskan mengapa CSP adalah metode yang tepat dibanding pencarian biasa.

Syarat minimal: 8 variabel, 3 nilai domain, 10 batasan biner, dan 2 batasan unary.

*Formulasi:*

In [ ]:
# Bagian B — Implementasi (30%): definisikan CSP Anda dengan kelas CSP, lalu selesaikan

In [ ]:
# Bagian C — Ketepatan hasil (25%): verifikasi dengan periksa(), dan bandingkan minimal 3 konfigurasi heuristik dalam satu tabel

### Bagian D — Kualitas kode dan penjelasan (20%)
Jelaskan hasil perbandingan pada Bagian C: konfigurasi mana yang paling hemat dan mengapa. Apa yang terjadi bila satu batasan ditambah sehingga masalah **tidak punya solusi**? Tunjukkan dengan kode dan jelaskan bagaimana program Anda memberi tahu pengguna.

*Penjelasan:*

### Refleksi
1. Dalam penjadwalan nyata, batasan mana yang bersifat **keras** (wajib) dan mana yang **lunak** (preferensi)? Bagaimana Anda memperlakukan batasan lunak?
2. Siapa yang dirugikan bila sebuah batasan penting lupa dimodelkan?

*Jawaban:*

---
**Pengumpulan A3:** `P07_NIM_Nama.ipynb` diunggah ke LMS. Notebook harus lolos *Restart and run all*.